# 🥉 Bronze Layer: Raw Data Ingestion

**Objective:** Ingest raw CSV supply chain datasets from Azure Data Lake Storage (ADLS) Gen2 and convert them into an optimized Delta format.

### Architecture Details
* **Ingestion Engine:** This pipeline utilizes **Databricks Autoloader (`cloudFiles`)** for scalable, incremental streaming. It only processes new files as they arrive in the storage container.
* **Schema Evolution:** We have enabled `schemaEvolutionMode = "addNewColumns"` to automatically detect and safely adapt to any structural changes in the upstream source data without breaking the pipeline.
* **Metadata Tracking:** Unity Catalog's native `_metadata.file_path` is extracted to track the exact origin of every ingested row for auditing purposes.

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Your External Location path handles all authentication
base_path = "abfss://iaspcontainer@iaspstorage.dfs.core.windows.net"

# The 6 core datasets for the supply chain pipeline
datasets = ["products", "inventory", "suppliers", "warehouses", "transactions", "shipments"]

print("Initializing Databricks Autoloader via Unity Catalog...")

for dataset in datasets:
    raw_path = f"{base_path}/raw/{dataset}/"
    bronze_path = f"{base_path}/bronze/{dataset}/"
    checkpoint_path = f"{base_path}/bronze/_checkpoints/{dataset}/"
    schema_path = f"{base_path}/bronze/_schemas/{dataset}/"
    
    print(f"Processing: {dataset}")
    
    try:
        # Read stream using Autoloader (cloudFiles)
        bronze_stream = (spark.readStream
                         .format("cloudFiles")
                         .option("cloudFiles.format", "csv")
                         .option("header", "true")
                         .option("cloudFiles.schemaLocation", schema_path)
                         .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
                         .load(raw_path))
        
        # THE FIX: Using Unity Catalog's native _metadata column
        processed_stream = (bronze_stream
                            .withColumn("ingestion_timestamp", current_timestamp())
                            .withColumn("source_file_name", col("_metadata.file_path")))
        
        # Write stream to Delta format
        (processed_stream.writeStream
                 .format("delta")
                 .option("checkpointLocation", checkpoint_path)
                 .outputMode("append")
                 .trigger(availableNow=True) # Optimized for Serverless
                 .option("path", bronze_path)
                 .toTable(f"bronze_{dataset}"))
                 
    except Exception as e:
        print(f"Error processing {dataset}: {e}")

print("Bronze Layer Ingestion Complete! Data is now in Delta format.")

Initializing Databricks Autoloader via Unity Catalog...
Processing: products
Processing: inventory
Processing: suppliers
Processing: warehouses
Processing: transactions
Processing: shipments
Bronze Layer Ingestion Complete! Data is now in Delta format.
